# 交互式问诊数据转换脚本 v2

将 SMHC MiroDiag 数据转换为交互式问诊训练格式

**v2 相比 v1 的关键修改：**
- System prompt 的输出规范部分重写，明确要求通过 tool call 进行问诊和诊断
- 诊断阶段格式统一为：`<think>推理</think>` + 调用 `do_diagnose` 工具，diagnosis 参数只放 `<box>ICD代码</box>`
- 移除了 system prompt 中直接输出 `<box>` 的自由文本格式（避免模型绕过 tool call）
- 与 SFT 数据 (`train_v2.json`) 的格式完全一致

In [1]:
import json
import pandas as pd
import re
import os
from typing import Dict, List, Any, Tuple

In [2]:
def extract_major_icd_codes(full_icd_code: str) -> "List[str]":
    """
    从完整 ICD 代码中提取大类代码（前3位）
    
    Args:
        full_icd_code: 完整的 ICD 代码，例如 'F20.400'
        
    Returns:
        大类代码列表，例如 ['F20']
    """
    if not full_icd_code:
        return []
    
    # 提取 F 开头的三位大类代码
    match = re.match(r'(F\d{2})', full_icd_code)
    if match:
        return [match.group(1)]
    return []


def create_interactive_diagnosis_prompt_v2() -> "List[Dict[str, str]]":
    """
    创建交互式问诊的 system prompt + 初始 user message (v2 - 与 SFT 格式对齐)
    
    v2 修改要点：
    1. 输出规范明确通过 tool call 输出，不使用自由文本格式
    2. 诊断阶段：<think>推理</think> 后调用 do_diagnose，diagnosis 参数只放 <box>ICD代码</box>
    3. 与 SFT 数据 train_v2.json 的格式完全一致
    
    Returns:
        包含 system prompt 和初始 user message 的消息列表
    """
    system_prompt = """你是一名精神卫生中心的临床心理科医师，进行专业问诊。
遵循 ICD-10 精神与行为障碍标准，先通过充分的共情式问诊收集信息，再进行诊断。

## 核心问诊原则

1. **建立治疗性关系**：
   - 在提问时表达对患者困扰的理解和共情
   - 使用支持性语言，如"我能理解这一定让您很困扰"、"听起来您经历了很艰难的时期"
   - 避免冷漠的连续提问，在关键信息后给予情感回应

2. **深入追问症状细节**：
   - 对患者描述的每个症状进行具体化：持续时间、频率、严重程度、触发因素
   - 询问症状的时间进程：何时开始、是否加重、是否有缓解期
   - 探查症状对日常功能的影响：工作、人际关系、自我照顾能力
   - 使用开放式问题引导患者详细描述，如"能详细说说吗"、"还有其他表现吗"

3. **验证症状真实性**：
   - 对患者自述的症状进行交叉验证：询问具体事例、客观表现
   - 观察症状描述的一致性和逻辑性
   - 区分患者的主观感受与客观症状

4. **鉴别诊断**：
   - 系统性排查相似疾病的鉴别点
   - 针对性询问关键区分症状（如抑郁与双相的既往躁狂史、焦虑与惊恐的发作模式）
   - 探查器质性因素：用药史、躯体疾病、物质使用
   - 评估共病可能：多个诊断可能并存时需要分别评估

## ICD-10 精神与行为障碍标准大类

请仅从以下ICD-10标准中的10种疾病中选择最符合的诊断大类以及进一步细分的ICD-10小类：

- F32 抑郁发作：情绪持续低落、兴趣/愉快感下降、精力不足；伴睡眠/食欲改变、自责/无价值感等；可轻/中/重度（重度可伴精神病性症状）；无既往躁狂/轻躁狂。
  *鉴别要点：需排除双相障碍（F31）、适应障碍（F43）、器质性情绪障碍*

- F41 其他焦虑障碍：恐慌发作或广泛性焦虑为主；过度担忧、紧张、心悸、胸闷、出汗、眩晕、濒死感/失控感；与特定情境无关或不成比例，造成显著痛苦/功能损害。
  *鉴别要点：需排除躯体疾病（如甲亢）、物质诱发、抑郁伴焦虑*

- F39.9 未特指的心境（情感）障碍：存在心境障碍证据，但资料不足以明确归入抑郁或双相等具体亚型时选用。

- F51 非器质性睡眠障碍：失眠、过度嗜睡、梦魇、昼夜节律紊乱等；非器质性原因；睡眠问题为主要主诉并致显著困扰/功能损害。
  *鉴别要点：需排除抑郁/焦虑继发失眠、睡眠呼吸暂停等器质性原因*

- F98 其他儿童和青少年行为与情绪障碍：多见于儿童期起病（如遗尿/遗粪、口吃、抽动相关习惯性问题等），以发育期特异表现为主。

- F42 强迫障碍：反复的强迫观念/行为，个体自知过度或不合理但难以抵抗，耗时或致显著困扰/损害。
  *鉴别要点：需排除精神分裂症的思维障碍、抑郁的反刍思维*

- F31 双相情感障碍：既往或目前存在躁狂/轻躁狂发作与抑郁发作的交替或混合；需有明确躁狂谱系证据。
  *鉴别要点：详细询问既往是否有情绪高涨、精力旺盛、睡眠需求减少、冲动行为等躁狂症状*

- F43 对严重应激反应和适应障碍：与明确应激事件有关；可为急性应激反应、PTSD或适应障碍；核心包含再体验、回避、警觉性增高或与应激源相关的情绪/行为改变。
  *鉴别要点：需确认应激源存在、症状与应激源的时间关联、是否超出正常应激反应*

- F45 躯体形式障碍：反复或多样躯体症状为主（如疼痛、心悸、胃肠不适等），检查难以找到足以解释的器质性原因或与病因不相称，显著痛苦/就诊反复。
  *鉴别要点：需详细了解躯体检查结果、症状与情绪的关联、疾病信念*

- F20 精神分裂症：在知觉、思维、情感及行为等方面的广泛障碍；常见持续性妄想、幻听、思维松弛/破裂、情感淡漠、阴性症状，病程≥1月（或依本地标准）。
  *鉴别要点：需评估现实检验能力、思维连贯性、情感协调性*

- Z71 咨询和医疗建议相关因素：包括心理咨询、健康教育、生活方式指导等，当患者主要需要咨询服务而非特定疾病治疗时使用。

## 问诊流程建议

1. **开场阶段**（1-2轮）：表达欢迎，了解主诉，建立初步信任
2. **症状探查阶段**（8-15轮）：深入了解核心症状及伴随症状，充分细节追问
3. **鉴别诊断阶段**（3-5轮）：针对性询问鉴别点，排除其他可能诊断
4. **功能评估阶段**（2-3轮）：评估症状对生活功能的影响
5. **诊断决策阶段**：综合信息，给出诊断

## 工具调用规范

你必须通过调用工具来完成问诊和诊断，每一轮输出的格式如下：

1) **问诊阶段**（信息不足时）：先在 <think> 标签中写出思考过程，然后调用 ask_patient 工具向患者提问。
   示例：
   <think>患者提到情绪低落，需要了解持续时间和严重程度</think>
   <tool_call>
   {"name": "ask_patient", "arguments": {"question": "我能理解您现在很不舒服。这种情绪低落的感觉大概持续多久了？"}}
   </tool_call>

2) **诊断阶段**（信息充分时）：先在 <think> 标签中写出完整的诊断推理，然后调用 do_diagnose 工具提交诊断。
   diagnosis 参数中只放 <box>ICD-10代码</box>，诊断推理写在 <think> 中。
   示例：
   <think>患者表现为持续情绪低落、兴趣减退、睡眠障碍，符合抑郁发作的诊断标准。排除双相障碍和适应障碍。</think>
   <tool_call>
   {"name": "do_diagnose", "arguments": {"diagnosis": "<box>F32.1</box>"}}
   </tool_call>

## 重要提示

- 当信息不足以排除其他疾病进行诊断时，必须继续调用 ask_patient 进行深入问诊
- 避免过早下诊断，确保收集到充分的鉴别信息
- 当确信可以诊断时，调用 do_diagnose，diagnosis 参数中必须包含 <box>ICD-10代码</box>
- ICD-10小类可多项，用分号分隔，如 <box>F32.1; F41.2</box>
- 保持温暖、专业的医患沟通风格，在专业性与人文关怀之间取得平衡"""
    
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "你好医生。"
        }
    ]


In [3]:
def make_interactive_map_fn(data_source: str, split: str):
    """
    创建交互式问诊数据转换函数
    
    Args:
        data_source: 数据源标识
        split: 'train' 或 'val'
    
    Returns:
        数据转换函数
    """
    def process_fn(data_item, idx):
        """处理单个数据项"""
        # 提取 patient_id 和诊断代码
        patient_id = data_item.get('patient_id')
        full_icd_code = data_item.get('DiagnosisCode', '')
        
        # 过滤无效数据
        if not patient_id or not full_icd_code:
            return None
        
        # 提取大类 ICD 代码
        major_icd_codes = extract_major_icd_codes(full_icd_code)
        
        if not major_icd_codes:
            return None
        
        # 创建 prompt（v2: 使用与 SFT 对齐的 system prompt）
        prompt = create_interactive_diagnosis_prompt_v2()
        
        # 构建转换后的数据
        converted_data = {
            "data_source": data_source,
            "prompt": prompt,
            "raw_prompt": prompt,
            "ability": "interactive_diagnosis",
            "reward_model": {
                "style": "rule",
                "ground_truth": major_icd_codes
            },
            "extra_info": {
                "split": split,
                "index": idx,
                "visit_number": patient_id,
                "full_icd_code": [full_icd_code],
                "diagnosis_name": data_item.get('Diagnosis', ''),
                "patient_id": patient_id,
                "tools_kwargs": {
                    "ask_patient": {
                        "create_kwargs": {
                            "patient_id": patient_id,
                            "patient_version": "v1",
                            "model_name": "moonshotai/kimi-k2-0905"
                        }
                    },
                    "do_diagnose": {
                        "create_kwargs": {
                            "patient_id": patient_id,
                            "ground_truth": major_icd_codes,
                            "patient_version": "v1",
                            "model_name": "moonshotai/kimi-k2-0905"
                        }
                    }
                }
            }
        }
        
        return converted_data
    
    return process_fn

In [4]:
def convert_to_interactive_diagnosis_parquet(
    train_file: str,
    output_dir: str,
    train_ratio: float = 0.9,
    max_samples: int = None
):
    """
    将数据转换为交互式问诊训练格式的 parquet 文件
    """
    data_source = "SMHC_RL_interactive_diagnosis"
    
    print(f"正在读取数据: {train_file}")
    with open(train_file, 'r', encoding='utf-8') as f:
        all_data = json.load(f)
    
    print(f"  读取到 {len(all_data)} 条数据")
    
    if max_samples:
        all_data = all_data[:max_samples]
        print(f"  限制样本数为 {max_samples}")
    
    # 去重：基于 patient_id
    print("正在按 patient_id 去重...")
    seen_ids = set()
    deduplicated_data = []
    
    for item in all_data:
        patient_id = item.get('patient_id', '')
        if patient_id and patient_id not in seen_ids:
            seen_ids.add(patient_id)
            deduplicated_data.append(item)
    
    print(f"去重后: {len(all_data)} -> {len(deduplicated_data)} 条")
    
    # 按比例划分训练集和验证集
    split_idx = int(len(deduplicated_data) * train_ratio)
    train_data = deduplicated_data[:split_idx]
    val_data = deduplicated_data[split_idx:]
    
    print(f"数据划分: 训练集 {len(train_data)} 条, 验证集 {len(val_data)} 条")
    
    # 创建转换函数
    train_map_fn = make_interactive_map_fn(data_source, 'train')
    val_map_fn = make_interactive_map_fn(data_source, 'val')
    
    # 转换训练数据
    print("正在转换训练数据...")
    converted_train = []
    filtered_train_count = 0
    for idx, item in enumerate(train_data):
        result = train_map_fn(item, idx)
        if result is not None:
            converted_train.append(result)
        else:
            filtered_train_count += 1
    
    # 转换验证数据
    print("正在转换验证数据...")
    converted_val = []
    filtered_val_count = 0
    for idx, item in enumerate(val_data):
        result = val_map_fn(item, idx)
        if result is not None:
            converted_val.append(result)
        else:
            filtered_val_count += 1
    
    print(f"过滤统计:")
    print(f"  训练集: 原始 {len(train_data)} 条, 过滤 {filtered_train_count} 条, 保留 {len(converted_train)} 条")
    print(f"  验证集: 原始 {len(val_data)} 条, 过滤 {filtered_val_count} 条, 保留 {len(converted_val)} 条")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 转换为 DataFrame 并保存为 parquet
    print("正在保存为 parquet 格式...")
    train_df = pd.DataFrame(converted_train)
    val_df = pd.DataFrame(converted_val)
    
    train_output_path = os.path.join(output_dir, 'train.parquet')
    val_output_path = os.path.join(output_dir, 'val.parquet')
    
    train_df.to_parquet(train_output_path, index=False)
    val_df.to_parquet(val_output_path, index=False)
    
    print(f"训练数据已保存至: {train_output_path}")
    print(f"验证数据已保存至: {val_output_path}")
    print(f"训练数据量: {len(converted_train)} 条, 验证数据量: {len(converted_val)} 条")
    
    if len(converted_train) > 0:
        print("\n样例数据 (训练集第一条):")
        print(json.dumps(converted_train[0], ensure_ascii=False, indent=2))
    
    return train_df, val_df

In [5]:
# 配置参数
TRAIN_FILE = "/tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json"
OUTPUT_DIR = "/tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy-v2"
TRAIN_RATIO = 0.9
MAX_SAMPLES = None  # 设置为 None 使用全部数据

# 执行转换
print("开始执行交互式问诊数据转换 (v2)...")
print("=" * 80)
print(f"输入文件: {TRAIN_FILE}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"训练集比例: {TRAIN_RATIO}")
if MAX_SAMPLES:
    print(f"限制样本数: {MAX_SAMPLES}")
print("=" * 80)

train_df, val_df = convert_to_interactive_diagnosis_parquet(
    train_file=TRAIN_FILE,
    output_dir=OUTPUT_DIR,
    train_ratio=TRAIN_RATIO,
    max_samples=MAX_SAMPLES
)

print("\n" + "=" * 80)
print("转换完成！")
print("=" * 80)

开始执行交互式问诊数据转换 (v2)...
输入文件: /tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json
输出目录: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy-v2
训练集比例: 0.9
正在读取数据: /tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json
  读取到 14000 条数据
正在按 patient_id 去重...
去重后: 14000 -> 13997 条
数据划分: 训练集 12597 条, 验证集 1400 条
正在转换训练数据...
正在转换验证数据...
过滤统计:
  训练集: 原始 12597 条, 过滤 371 条, 保留 12226 条
  验证集: 原始 1400 条, 过滤 43 条, 保留 1357 条
正在保存为 parquet 格式...
训练数据已保存至: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy-v2/train.parquet
验证数据已保存至: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy-v2/val.parquet
训练数据量: 12226 条, 验证数据量: 1357 条

样例数据 (训练集第一条):
{
  "data_source": "SMHC_RL_interactive_diagnosis",
  "prompt": [
    {
      "role": "system",
      "content": "你是一名精神卫生中心的临床心理科医师，进行专业问诊。\n遵循 ICD-10 精神与行为障碍标准，先通过充分的共情式问诊收集信息，再进行诊断。\n\n## 核心问诊原则\n\n1

In [6]:
# 验证生成的数据
print("数据验证:")
print("=" * 80)

print(f"训练集数据量: {len(train_df)}")
print(f"验证集数据量: {len(val_df)}")
print(f"数据列名: {train_df.columns.tolist()}")

# 检查数据结构
sample = train_df.iloc[0]
print(f"\n训练集第一条数据:")
print(f"- data_source: {sample['data_source']}")
print(f"- ability: {sample['ability']}")
print(f"- prompt: 包含 {len(sample['prompt'])} 个消息")
print(f"- reward_model: {sample['reward_model']}")

# 显示 system prompt 的关键部分
sp = sample['prompt'][0]['content']
print(f"\n[v2 system prompt 关键特征]")
print(f"  - 包含 '工具调用规范': {'工具调用规范' in sp}")
print(f"  - 包含 'ask_patient' 示例: {'ask_patient' in sp}")
print(f"  - 包含 'do_diagnose' 示例: {'do_diagnose' in sp}")
print(f"  - 包含旧格式 '输出规范（只选其一）': {'输出规范（只选其一）' in sp}")
print(f"  - System prompt 长度: {len(sp)} 字符")

# 统计诊断代码分布
print("\n诊断代码分布 (训练集 前10):")
train_diagnosis = train_df['reward_model'].apply(lambda x: str(x['ground_truth']))
print(train_diagnosis.value_counts().head(10))

print("\n诊断代码分布 (验证集 前10):")
val_diagnosis = val_df['reward_model'].apply(lambda x: str(x['ground_truth']))
print(val_diagnosis.value_counts().head(10))

# 数据质量检查
print(f"\n数据质量检查:")
print(f"训练集空值: {train_df.isnull().sum().sum()}")
print(f"验证集空值: {val_df.isnull().sum().sum()}")

print("\n" + "=" * 80)
print("验证完成！数据已准备好用于交互式问诊训练 (v2)")
print("=" * 80)

数据验证:
训练集数据量: 12226
验证集数据量: 1357
数据列名: ['data_source', 'prompt', 'raw_prompt', 'ability', 'reward_model', 'extra_info']

训练集第一条数据:
- data_source: SMHC_RL_interactive_diagnosis
- ability: interactive_diagnosis
- prompt: 包含 2 个消息
- reward_model: {'style': 'rule', 'ground_truth': ['F41']}

[v2 system prompt 关键特征]
  - 包含 '工具调用规范': True
  - 包含 'ask_patient' 示例: True
  - 包含 'do_diagnose' 示例: True
  - 包含旧格式 '输出规范（只选其一）': False
  - System prompt 长度: 2676 字符

诊断代码分布 (训练集 前10):
reward_model
['F32']    4576
['F41']    4432
['F39']     732
['F51']     450
['F98']     367
['F42']     321
['F43']     133
['F28']     106
['F31']     106
['F45']     101
Name: count, dtype: int64

诊断代码分布 (验证集 前10):
reward_model
['F32']    503
['F41']    459
['F39']     93
['F51']     61
['F98']     42
['F42']     28
['F45']     15
['F43']     14
['F31']     13
['F28']     11
Name: count, dtype: int64

数据质量检查:
训练集空值: 0
验证集空值: 0

验证完成！数据已准备好用于交互式问诊训练 (v2)
